# probing-trusted-monitors -- Kaggle GPU runnerRuns the GPU-bound stages (0-3) of the project and zips the results for download.Stages 4-6 are CPU-cheap and are meant to run locally on the imported archive.See `KAGGLE.md` in the repo for the full procedure.

In [ ]:
# ---- configuration -------------------------------------------------------REPO_URL     = "https://github.com/ShreyasVR2545/probing-trusted-monitors.git"BRANCH       = "main"STAGES       = ["0", "1", "2", "3"]   # GPU stages; 4-6 run locallySMOKE        = True                    # run this once before a long sessionFORCE_MODEL  = None                    # e.g. "Qwen/Qwen2.5-7B-Instruct"N_TOTAL      = None                    # e.g. 1200 to override configs/base.yaml

In [ ]:
import subprocess, sys, os, shutilos.chdir("/kaggle/working")if os.path.exists("probing-trusted-monitors"):    shutil.rmtree("probing-trusted-monitors")subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL], check=True)os.chdir("/kaggle/working/probing-trusted-monitors")print(subprocess.run(["git","rev-parse","HEAD"],capture_output=True,text=True).stdout.strip())

In [ ]:
# Kaggle images already carry torch+CUDA; only install what is missing so we do# not disturb the preinstalled, GPU-matched torch build.import subprocess, syssubprocess.run([sys.executable,"-m","pip","install","-q",                "transformers==4.57.1","accelerate==1.11.0","bitsandbytes==0.48.2",                "datasets==4.4.1","huggingface_hub==0.36.0","pyyaml"], check=False)import torch, transformersprint("torch",torch.__version__,"cuda",torch.cuda.is_available())if torch.cuda.is_available():    p=torch.cuda.get_device_properties(0)    print(p.name, round(p.total_memory/1024**3,1),"GB cc",p.major,p.minor,          "bf16",torch.cuda.is_bf16_supported())print("transformers",transformers.__version__)

In [ ]:
# Hardware detection writes configs/hardware.yaml, which every stage reads.import subprocess, syssubprocess.run([sys.executable,"-m","src.utils.hardware"], check=True)print(open("HARDWARE.md").read())

In [ ]:
# Optional overrides applied on top of the detected plan.import yamlif FORCE_MODEL:    hw = yaml.safe_load(open("configs/hardware.yaml"))    hw["plan"]["primary_model"] = FORCE_MODEL    # A 16GB Kaggle GPU runs 7B unquantised; keep bf16 only if truly supported.    import torch    hw["plan"]["quantization"] = "none" if torch.cuda.get_device_properties(0).total_memory/1024**3 >= 15 else "4bit"    if not torch.cuda.is_bf16_supported():        hw["plan"]["dtype"] = "float16"    yaml.safe_dump(hw, open("configs/hardware.yaml","w"), sort_keys=False)    print("forced model:", FORCE_MODEL, "| quant:", hw["plan"]["quantization"],          "| dtype:", hw["plan"]["dtype"])if N_TOTAL:    base = yaml.safe_load(open("configs/base.yaml"))    base["data"]["n_total"] = N_TOTAL    # Kaggle hosts are stable and not the machine that was hard-resetting, so the    # cooldown pacing is unnecessary there.    base["power"]["enabled"] = False    yaml.safe_dump(base, open("configs/base.yaml","w"), sort_keys=False)    print("n_total:", N_TOTAL)

In [ ]:
import subprocess, sys, zipfile, os, timedef archive():    """Write results.zip incrementally so a lost session does not lose everything."""    out = "/kaggle/working/results.zip"    tmp = out + ".tmp"    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:        for root, _, files in os.walk("results"):            if os.sep + "raw" in root or os.sep + "index" in root:                continue          # re-downloadable, and not ours to redistribute            for f in files:                p = os.path.join(root, f)                zf.write(p, os.path.relpath(p, "."))    os.replace(tmp, out)    print(f"  -> results.zip {os.path.getsize(out)/1024**2:.1f} MB")for stage in STAGES:    print(f"{'='*70}  STAGE {stage}{'='*70}", flush=True)    cmd = [sys.executable, "scripts/run_all.py", "--stage", stage, "--pass-gates"]    if SMOKE:        cmd.append("--smoke")    t0 = time.time()    rc = subprocess.run(cmd).returncode    print(f"stage {stage} exit={rc} in {time.time()-t0:.0f}s", flush=True)    archive()    if rc != 0:        raise SystemExit(f"stage {stage} failed with exit code {rc}")print("All stages complete.")

In [ ]:
# Gate 1 report -- read this before committing to a long run.import json, osp = "results/signal_check/gate1_report.json"if os.path.exists(p):    r = json.load(open(p))    for k, v in r.items():        if k in ("tie_table","histogram"): continue        print(f"{k}: {json.dumps(v, default=str)[:300]}")else:    print("no gate1 report (stage 1 not run)")

## Download`results.zip` is in the **Output** panel on the right. Then, locally:```bashpython scripts/run_all.py --import-results results.zippython scripts/run_all.py --stage 4    # transfer, GATE 2python scripts/run_all.py --stage 5python scripts/run_all.py --stage 6```